In [1]:
import pandas as pd

# Define a raw data dataframe
raw_df = pd.read_csv("C:/Projetos/biofuels-sciml/data/raw/toy_problem_raw_dataset.csv")
raw_df

,substance_1,substance_2,x_1,x_2,T,gamma_1,gamma_2
0,hexane,heptane,0.0,1.0,371.549,0.991430,1.000000
1,hexane,heptane,0.1,0.9,367.477,0.992940,0.999920
2,hexane,heptane,0.2,0.8,363.713,0.994320,0.999670
3,hexane,heptane,0.3,0.7,360.232,0.995570,0.999250
4,hexane,heptane,0.4,0.6,357.008,0.996690,0.998650
...,...,...,...,...,...,...,...
281,dodecane,octane,0.6,0.4,432.620,0.990977,0.974314
282,dodecane,octane,0.7,0.3,442.541,0.995141,0.966776
283,dodecane,octane,0.8,0.2,454.858,0.997931,0.958714
284,dodecane,octane,0.9,0.1,470.327,0.999504,0.950231


In [2]:
from ugropy import Groups


# Define functions to calculate r and q from UNIFAC
def calculate_r(substance: str) -> float:
    # Load unifac parameters
    unifac_parameters = pd.read_csv(
        "C:/Projetos/biofuels-sciml/data/unifac_parameters/unifac_r_and_q.csv"
    )
    unifac_r_values = unifac_parameters.set_index("Subgroup")["R (volume)"].to_dict()

    init_substance = Groups(substance)
    substance_group = init_substance.unifac.subgroups

    # Calculate r value
    initial_r = 0
    for key in substance_group.keys():
        initial_r += substance_group[key] * unifac_r_values[key]

    return round(initial_r, 4)


def calculate_q(substance: str) -> float:
    # Load unifac parameters
    unifac_parameters = pd.read_csv(
        "C:/Projetos/biofuels-sciml/data/unifac_parameters/unifac_r_and_q.csv"
    )
    unifac_q_values = unifac_parameters.set_index("Subgroup")["Q (area)"].to_dict()

    init_substance = Groups(substance)
    substance_group = init_substance.unifac.subgroups

    # Calculate r value
    initial_q = 0
    for key in substance_group.keys():
        initial_q += substance_group[key] * unifac_q_values[key]

    return round(initial_q, 4)

In [3]:
# Create new dataset
substance_columns = [col for col in raw_df.columns if col.startswith("substance")]
num_substances = len(substance_columns)

for index, col in enumerate(substance_columns, start=1):
    raw_df[f"r_{index}"] = raw_df[col].apply(calculate_r)
    raw_df[f"q_{index}"] = raw_df[col].apply(calculate_q)

# Drop string columns
raw_df.drop(substance_columns, axis=1, inplace=True)

# Reorder the columns
new_order = [
    col
    for i in range(1, len(substance_columns) + 1)
    for col in [f"r_{i}", f"q_{i}", f"x_{i}"]
]
new_order += ["T"]
new_order += [
    col for i in range(1, len(substance_columns) + 1) for col in [f"gamma_{i}"]
]

processed_df = raw_df[new_order]

# Save table
processed_df.to_csv(
    "C:/Projetos/biofuels-sciml/data/processed/toy_problem_input_dataset.csv", index=False
)